In [0]:
spark.sql("USE default")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bangjie_model1_predictions (
    grid DOUBLE,
    points DOUBLE,
    laps DOUBLE,
    year DOUBLE,
    top10 INT,
    prediction DOUBLE,
    probability STRING
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS bangjie_model2_predictions (
    grid DOUBLE,
    points DOUBLE,
    laps DOUBLE,
    year DOUBLE,
    top10 INT,
    prediction DOUBLE,
    probability STRING
)
""")

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.sql("SHOW DATABASES").show()

In [0]:
spark.sql("SHOW TABLES IN default").show()

In [0]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
%run ./etl_pipeline

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
tables = spark.sql("SHOW TABLES").toPandas()

tables[tables['tableName'].str.contains('result', case=False)]

In [0]:
spark.sql("SHOW TABLES").show(100, truncate=False)

In [0]:
spark.sql("SELECT current_database()").show()

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
%run ./etl_pipeline

In [0]:
for name in dir():
    if any(x in name.lower() for x in ["race", "result", "driver", "constructor"]):
        print(name)

In [0]:
print(type(results))
print(type(races))

In [0]:
%run ./etl_pipeline

In [0]:
%run "./etl_pipeline"

In [0]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import mlflow
import mlflow.spark

spark.sql("USE default")

df = results.join(races, "raceId")

model_df = df.select(
    col("grid").cast("double"),
    col("points").cast("double"),
    col("laps").cast("double"),
    col("year").cast("double"),
    col("positionOrder").cast("double")
).dropna()

model_df = model_df.withColumn(
    "top10",
    when(col("positionOrder") <= 10, 1).otherwise(0)
).select("grid", "points", "laps", "year", "top10")

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(
    inputCols=["grid", "points", "laps", "year"],
    outputCol="features"
)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="top10",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="top10",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="top10",
    predictionCol="prediction",
    metricName="f1"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="top10",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="top10",
    predictionCol="prediction",
    metricName="weightedRecall"
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="top10",
    maxIter=10,
    regParam=0.1
)

pipeline_lr = Pipeline(stages=[assembler, lr])

with mlflow.start_run(run_name="bangjie_logistic_regression"):
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("regParam", 0.1)

    lr_model = pipeline_lr.fit(train_df)
    lr_predictions = lr_model.transform(test_df)

    mlflow.log_metric("auc", evaluator_auc.evaluate(lr_predictions))
    mlflow.log_metric("accuracy", evaluator_acc.evaluate(lr_predictions))
    mlflow.log_metric("f1", evaluator_f1.evaluate(lr_predictions))
    mlflow.log_metric("precision", evaluator_precision.evaluate(lr_predictions))
    mlflow.log_metric("recall", evaluator_recall.evaluate(lr_predictions))

    mlflow.spark.log_model(lr_model, "logistic_regression_model")

    lr_predictions.select(
        "grid", "points", "laps", "year", "top10", "prediction", "probability"
    ).withColumn(
        "probability", col("probability").cast("string")
    ).write.mode("overwrite").saveAsTable("bangjie_model1_predictions")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="top10",
    numTrees=50,
    maxDepth=5,
    seed=42
)

pipeline_rf = Pipeline(stages=[assembler, rf])

with mlflow.start_run(run_name="bangjie_random_forest"):
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)

    rf_model = pipeline_rf.fit(train_df)
    rf_predictions = rf_model.transform(test_df)

    mlflow.log_metric("auc", evaluator_auc.evaluate(rf_predictions))
    mlflow.log_metric("accuracy", evaluator_acc.evaluate(rf_predictions))
    mlflow.log_metric("f1", evaluator_f1.evaluate(rf_predictions))
    mlflow.log_metric("precision", evaluator_precision.evaluate(rf_predictions))
    mlflow.log_metric("recall", evaluator_recall.evaluate(rf_predictions))

    mlflow.spark.log_model(rf_model, "random_forest_model")

    rf_predictions.select(
        "grid", "points", "laps", "year", "top10", "prediction", "probability"
    ).withColumn(
        "probability", col("probability").cast("string")
    ).write.mode("overwrite").saveAsTable("bangjie_model2_predictions")

spark.table("bangjie_model1_predictions").show(10)
spark.table("bangjie_model2_predictions").show(10)
spark.sql("SHOW TABLES").show(truncate=False)